# DA-BiGRU-CNN production training run (Colab GPU)

**Project.** `predictive-mcdm-defi` - HSE FCS DeFi-Strategies Project 2 (18 May - 14 June 2026).

**Goal of this notebook.** End-to-end production training of the dual-branch forecaster (`forecaster.model.DABiGRUCNNForecaster`) on real Aave V3 + Compound V3 USDC supply-rate data. Outputs are a PyTorch checkpoint, an ONNX export, and a JSON metrics blob - all written back to Google Drive so they survive Colab session expiry.

**Workflow.** Run `python -m scripts.prepare_colab_artifacts` locally first; upload the resulting `predictive-mcdm-defi-artifacts.zip` to `MyDrive/predictive-mcdm-defi/`; then **Runtime -> Change runtime type -> GPU** (A100 or H100 preferred), then **Run All**.

**Expected runtime.** 30-45 min on H100, ~60 min on A100, ~90+ min on T4. Plan: 15 epochs (PROJECT_2_PLAN.md S4.4), `sequence_length=168` (7-day), `forecast_horizon=12h`, `batch_size=64`, AdamW lr=2e-3 wd=0.01, cosine annealing, early stopping patience=5.

**Outputs (copied to Drive).**

| Artifact                                | Purpose                              |
|-----------------------------------------|--------------------------------------|
| `dual_branch_kink.onnx`                 | runtime forecaster (used by backtest)|
| `da_bigru_cnn.pt`                       | PyTorch checkpoint (re-exportable)   |
| `metrics.json`                          | val wPearson, dir-acc, R^2 per protocol |

**Defensive design.** Every cell prints its starting state and gates on a `READY_<step>` boolean from the previous cell - if a prior cell failed the rest short-circuit with a clear message rather than burning GPU minutes on bad inputs.

## 1. Colab detection + dependency install

We do NOT touch `torch` here - the Colab base image's torch is already linked against the right CUDA driver and re-installing from PyPI would silently downgrade to a CPU wheel. We only add: pure-Python deps (mlflow, statsmodels, etc.) and the pinned `fractal-defi==1.3.2` from the git tag (errata 1: v1.3.2 is the latest tagged release; v1.4.0 is unreleased).

In [ ]:
READY_install = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"IN_COLAB = {IN_COLAB}")

if IN_COLAB:
    import subprocess, sys
    extras = [
        "mlflow",
        "catboost",
        "statsmodels",
        "onnx",
        "onnxruntime",
        "python-dotenv",
        "pyarrow",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *extras])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "fractal-defi @ git+https://github.com/Logarithm-Labs/fractal-defi.git@v1.3.2",
    ])
    print("[install] dependencies OK")
else:
    print("[install] local mode - assuming .venv already has fractal-defi==1.3.2 + torch")

READY_install = True

## 2. GPU detection (fail fast on CPU)

If `torch.cuda.is_available()` is False, halt immediately with an actionable message - the full 15-epoch run on CPU would take ~ 8 hours and the Colab session would expire first. Also note (CLAUDE.md): `import torch` BEFORE numpy/pandas on Windows-local to avoid the c10.dll load-order bug. On Colab Linux this ordering doesn't matter but we keep it for parity.

In [ ]:
READY_gpu = False
assert READY_install, "Cell 1 (install) failed - fix that first."

import torch  # MUST come before numpy/pandas (CLAUDE.md DLL-order note)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. Fix: Runtime -> Change runtime type -> Hardware accelerator -> GPU "
        "(A100 or H100 strongly preferred; T4 also works but ~3x slower). "
        "For a CPU smoke test use the local 03_forecaster_training.ipynb instead."
    )

device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print(f"GPU:        {props.name}")
print(f"VRAM:       {props.total_memory / 1e9:.1f} GB")
print(f"CC:         {props.major}.{props.minor}")
print(f"torch:      {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")

# Enable TF32 matmul on Ampere+ for ~1.5x GRU speedup (no accuracy impact for our scale).
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
READY_gpu = True

## 3. Mount Google Drive (idempotent)

We mount at `/content/drive`. `drive.mount(force_remount=False)` is a no-op if the drive is already mounted, so re-running this cell mid-session does not re-trigger the OAuth flow. The standard bundle layout we expect on Drive is:

```
MyDrive/predictive-mcdm-defi/
    predictive-mcdm-defi-artifacts.zip   <- produced by scripts/prepare_colab_artifacts.py
    trained_models/                      <- created by this notebook on first run
    mlruns/                              <- MLflow tracking store
```

In [ ]:
READY_drive = False
assert READY_gpu, "Cell 2 (GPU check) failed - fix that first."

import subprocess, sys as _sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/predictive-mcdm-defi")
DRIVE_OK = False          # True only if Drive actually mounted + folder exists
BUNDLE_PATH = None
_BUNDLE_NAME = "predictive-mcdm-defi-artifacts.zip"

# === Headless bundle-in for the VS Code -> Colab runtime path ===============
# drive.mount()'s browser OAuth widget does not exist when this notebook is
# driven from VS Code's Jupyter client, so we fetch the bundle from a SHARED
# Google Drive FOLDER via gdown - no mount, no OAuth, runs on the VM.
# The folder is shared "Anyone with the link"; the zip lives inside it.
BUNDLE_GDRIVE_FOLDER = "https://drive.google.com/drive/folders/1k_nZZdsMfD3umj5pMZopIdsxqobtSzhB?usp=sharing"
BUNDLE_GDRIVE_ID = ""   # optional: a direct FILE id instead of the folder
# ===========================================================================

_CONTENT_CANDIDATES = [
    Path("/content") / _BUNDLE_NAME,
    Path("/content/drive/MyDrive/predictive-mcdm-defi") / _BUNDLE_NAME,
]

# 1) Drive mount - best effort (works on Colab-web "Run All", not from VS Code)
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_OK = DRIVE_ROOT.exists()
        print(f"[drive] mount OK; project folder {'found' if DRIVE_OK else 'MISSING'}")
        if DRIVE_OK and (DRIVE_ROOT / _BUNDLE_NAME).exists():
            BUNDLE_PATH = DRIVE_ROOT / _BUNDLE_NAME
    except Exception as exc:  # noqa: BLE001
        print(f"[drive] mount unavailable ({type(exc).__name__}: {exc}); "
              "using gdown (expected from VS Code -> Colab runtime).")
else:
    print("[drive] non-Colab mode - using local file paths")

# 2) gdown - shared folder (preferred) or a direct file id
if IN_COLAB and BUNDLE_PATH is None and (BUNDLE_GDRIVE_FOLDER.strip()
                                         or BUNDLE_GDRIVE_ID.strip()):
    try:
        subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q",
                               "gdown>=5.1.0"])
        import gdown
        _dst = Path("/content") / _BUNDLE_NAME
        if BUNDLE_GDRIVE_ID.strip():
            gdown.download(id=BUNDLE_GDRIVE_ID.strip(), output=str(_dst),
                           quiet=False)
        else:
            _outdir = Path("/content/_bundle_dl")
            gdown.download_folder(url=BUNDLE_GDRIVE_FOLDER.strip(),
                                  output=str(_outdir), quiet=False,
                                  use_cookies=False)
            _zips = sorted(_outdir.rglob("*.zip"))
            if _zips:
                # prefer the canonical name if present, else first zip
                _named = [z for z in _zips if z.name == _BUNDLE_NAME]
                (_named[0] if _named else _zips[0]).replace(_dst)
        if _dst.exists() and _dst.stat().st_size > 100_000:
            BUNDLE_PATH = _dst
            print(f"[gdown] fetched bundle -> {_dst} "
                  f"({_dst.stat().st_size/1e6:.2f} MB)")
        else:
            print("[gdown] no/too-small file - verify the folder is shared "
                  "'Anyone with the link'.")
    except Exception as exc:  # noqa: BLE001
        print(f"[gdown] failed ({exc}); falling back to /content lookup.")

# 3) plain /content fallback (manual upload onto the VM)
if IN_COLAB and BUNDLE_PATH is None:
    for _c in _CONTENT_CANDIDATES:
        if _c.exists():
            BUNDLE_PATH = _c
            break

if IN_COLAB and BUNDLE_PATH is None:
    raise RuntimeError(
        "Artifacts bundle not found. Resolution order tried: Drive-mount, "
        "gdown(folder/id), /content. Fix one of:\n"
        f"  - confirm the shared folder is 'Anyone with the link' and contains "
        f"{_BUNDLE_NAME}\n"
        f"  - or set BUNDLE_GDRIVE_ID to a direct file id\n"
        f"  - or upload {_BUNDLE_NAME} to /content on the VM\n"
        "Regenerate locally via:  python -m scripts.prepare_colab_artifacts"
    )

if IN_COLAB:
    print(f"[bundle] using {BUNDLE_PATH} "
          f"({BUNDLE_PATH.stat().st_size / 1e6:.2f} MB)   DRIVE_OK={DRIVE_OK}")
READY_drive = True

## 4. Unpack bundle + register Python paths

Extract the zip to `/content/predictive-mcdm-defi/` and prepend that path to `sys.path` so `from forecaster.model import ...` resolves. We also `os.chdir` to the project root so any relative paths inside Trainer (e.g. `forecaster/trained_models/`) behave the same on Colab as locally.

In [ ]:
READY_unpack = False
assert READY_drive, "Cell 3 (Drive mount) failed - fix that first."

import os, sys, zipfile
from pathlib import Path

if IN_COLAB:
    PROJECT_ROOT = Path("/content/predictive-mcdm-defi")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(BUNDLE_PATH) as zf:
            zf.extractall(PROJECT_ROOT)
        members = list(PROJECT_ROOT.rglob("*.py"))
        print(f"[unpack] extracted {len(members)} .py files + data into {PROJECT_ROOT}")
    except Exception as exc:  # noqa: BLE001
        raise RuntimeError(
            f"Failed to extract bundle {BUNDLE_PATH}: {exc}. "
            f"Re-upload predictive-mcdm-defi-artifacts.zip and re-run."
        ) from exc
else:
    cand = Path.cwd()
    PROJECT_ROOT = None
    for p in [cand, *cand.parents]:
        if (p / "PROJECT_2_PLAN.md").exists():
            PROJECT_ROOT = p
            break
    if PROJECT_ROOT is None:
        raise RuntimeError("Could not locate repo root (PROJECT_2_PLAN.md not found in parents).")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"[paths] cwd = {os.getcwd()}")

REQUIRED = [
    "data/cached/joined_clean.parquet",
    "data/cached/kink_params.json",
    "data/features.py",
    "forecaster/model.py",
    "forecaster/train.py",
    "forecaster/losses.py",
    "forecaster/export_onnx.py",
]
missing = [r for r in REQUIRED if not (PROJECT_ROOT / r).exists()]
if missing:
    raise RuntimeError("Bundle is missing required files: " + ", ".join(missing))

print("[paths] all required artifacts present")
READY_unpack = True

## 5. Load data + apply feature pipeline

Load `joined_clean.parquet` (one row per hour, both protocols), load kink params, run `data.features.extract_features` to get residuals, cross-protocol spreads, and time-of-day cyclical encoding.

In [ ]:
READY_data = False
assert READY_unpack, "Cell 4 (unpack) failed - fix that first."

import json
import numpy as np
import pandas as pd

from data.features import AaveKinkParams, CompoundKinkParams, extract_features

DATA_PATH = PROJECT_ROOT / "data" / "cached" / "joined_clean.parquet"
KINK_PATH = PROJECT_ROOT / "data" / "cached" / "kink_params.json"

DATA = pd.read_parquet(DATA_PATH)
if not isinstance(DATA.index, pd.DatetimeIndex):
    raise RuntimeError(
        f"joined_clean.parquet must have a DatetimeIndex, got {type(DATA.index).__name__}"
    )
if DATA.index.tz is None:
    DATA.index = DATA.index.tz_localize("UTC")
print(f"[data] {len(DATA):,} rows  {DATA.index[0]}  ->  {DATA.index[-1]}")
print(f"[data] columns: {list(DATA.columns)}")

kp = json.loads(KINK_PATH.read_text())
KINK_AAVE = AaveKinkParams(**kp["aave"])
KINK_COMPOUND = CompoundKinkParams(**kp["compound"])
print(f"[kink] aave={KINK_AAVE}")
print(f"[kink] compound={KINK_COMPOUND}")

FEATS = extract_features(DATA, KINK_AAVE, KINK_COMPOUND).dropna()
print(f"[features] panel shape: {FEATS.shape}")
READY_data = True

## 6. Chronological train / val / test split (PROJECT_2_PLAN.md S4.1)

Strict calendar split, no shuffling:

| Split | Window                  | Purpose                                  |
|-------|-------------------------|------------------------------------------|
| train | 2024-11-01 - 2025-08-31 | forecaster training                      |
| val   | 2025-09-01 - 2025-12-31 | early-stop + hyperparameter selection    |
| test  | 2026-01-01 - 2026-04-30 | final unseen metric (held out from train) |

If your data window is narrower than the planned window (e.g. the parquet only covers part of 2026), the cell adapts but warns.

In [ ]:
READY_split = False
assert READY_data, "Cell 5 (data) failed - fix that first."

TRAIN_END = pd.Timestamp("2025-09-01", tz="UTC")
VAL_END   = pd.Timestamp("2026-01-01", tz="UTC")
TEST_END  = pd.Timestamp("2026-05-01", tz="UTC")

FEATS_TRAIN = FEATS.loc[FEATS.index <  TRAIN_END]
FEATS_VAL   = FEATS.loc[(FEATS.index >= TRAIN_END) & (FEATS.index < VAL_END)]
FEATS_TEST  = FEATS.loc[(FEATS.index >= VAL_END)   & (FEATS.index < TEST_END)]

for name, slc in [("train", FEATS_TRAIN), ("val", FEATS_VAL), ("test", FEATS_TEST)]:
    if len(slc) < 200:
        print(f"[warn] {name} split has only {len(slc)} rows - sequence_length=168 may be too long")
    else:
        print(f"[split] {name}: {len(slc):>5} rows   {slc.index[0]} -> {slc.index[-1]}")

if len(FEATS_TRAIN) == 0 or len(FEATS_VAL) == 0:
    raise RuntimeError(
        "Train or val split is empty - check the parquet's date range. "
        f"Got {FEATS.index[0]} -> {FEATS.index[-1]} but expected coverage of "
        f"at least 2024-11 to 2025-12 for the planned split."
    )
READY_split = True

## 7. Build Datasets + DataLoaders

`sequence_length=168` (7-day rolling window), `forecast_horizon=12` (12h ahead), `batch_size=64`. `pin_memory=True` gives ~30% faster host->GPU transfer on A100/H100. We use `num_workers=2` to match Colab's 2 vCPU.

In [ ]:
READY_loaders = False
assert READY_split, "Cell 6 (split) failed - fix that first."

from torch.utils.data import DataLoader
from forecaster.train import DABiGRUCNNDataset

SEQ_LEN = 168
HORIZON = 12
BATCH = 64

ds_train = DABiGRUCNNDataset(FEATS_TRAIN, KINK_AAVE, KINK_COMPOUND,
                             input_window=SEQ_LEN, forecast_horizon=HORIZON)
ds_val   = DABiGRUCNNDataset(FEATS_VAL,   KINK_AAVE, KINK_COMPOUND,
                             input_window=SEQ_LEN, forecast_horizon=HORIZON)
ds_test  = (DABiGRUCNNDataset(FEATS_TEST, KINK_AAVE, KINK_COMPOUND,
                              input_window=SEQ_LEN, forecast_horizon=HORIZON)
            if len(FEATS_TEST) > SEQ_LEN + HORIZON else None)

train_loader = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  drop_last=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = (DataLoader(ds_test,  batch_size=BATCH, shuffle=False,
                           num_workers=2, pin_memory=True)
                if ds_test is not None else None)

print(f"[loaders] train batches = {len(train_loader)} (samples {len(ds_train)})")
print(f"[loaders] val   batches = {len(val_loader)} (samples {len(ds_val)})")
if test_loader is not None:
    print(f"[loaders] test  batches = {len(test_loader)} (samples {len(ds_test)})")
else:
    print("[loaders] test split too small for sequence_length - skipping held-out test metrics")
READY_loaders = True

## 8. Training (15 epochs, AdamW, cosine annealing, early stop=5)

Production settings per PROJECT_2_PLAN.md S4.4 / S5.1 defaults: hidden=64 per branch, BiGRU(2 layers), CNN kernels=[3,5,7], dropout=0.1, lr=2e-3, wd=0.01. The trainer logs to MLflow if available (file-store under `MyDrive/predictive-mcdm-defi/mlruns/`); failures there are non-fatal.

In [ ]:
READY_train = False
assert READY_loaders, "Cell 7 (loaders) failed - fix that first."

import time
from forecaster.model import DABiGRUCNNForecaster, ForecasterConfig
from forecaster.train import TrainConfig, Trainer

# Output dir: Drive if it actually mounted, else a VM-local dir. With
# DRIVE_OK False (the VS Code -> Colab path) artifacts go to /content and
# Cell 11 zips them for one-shot retrieval - nothing depends on Drive.
if IN_COLAB and DRIVE_OK:
    CKPT_DIR = DRIVE_ROOT / "trained_models"
elif IN_COLAB:
    CKPT_DIR = PROJECT_ROOT / "trained_models"          # /content/predictive-mcdm-defi/trained_models
else:
    CKPT_DIR = PROJECT_ROOT / "forecaster" / "trained_models"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CKPT_DIR / "da_bigru_cnn.pt"
print(f"[out] checkpoint dir = {CKPT_DIR}  (DRIVE_OK={DRIVE_OK if IN_COLAB else 'n/a'})")

MLFLOW_EXPERIMENT = "defi-forecast-colab-production"
try:
    import mlflow
    if IN_COLAB:
        ml_dir = (DRIVE_ROOT / "mlruns") if DRIVE_OK else (PROJECT_ROOT / "mlruns")
        ml_dir.mkdir(parents=True, exist_ok=True)
        mlflow.set_tracking_uri(f"file://{ml_dir}")
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    print(f"[mlflow] tracking: {mlflow.get_tracking_uri()}")
except Exception as exc:  # noqa: BLE001
    print(f"[mlflow] disabled: {exc}")
    MLFLOW_EXPERIMENT = None

model_cfg = ForecasterConfig(
    branch_a_hidden=64,
    branch_b_hidden=64,
    head_hidden=64,
    branch_a_layers=2,
    branch_b_layers=2,
    branch_b_cnn_kernels=(3, 5, 7),
    dropout=0.1,
    sequence_length=SEQ_LEN,
    forecast_horizon=HORIZON,
)
train_cfg = TrainConfig(
    input_window=SEQ_LEN,
    forecast_horizon=HORIZON,
    batch_size=BATCH,
    lr=2e-3,
    weight_decay=0.01,
    grad_clip=1.0,
    max_epochs=15,
    patience=5,
    num_workers=2,
    device="cuda",
    alpha=0.4, beta=0.5, gamma=0.1, quantile_q=0.9,
    n_splits=1,
    checkpoint_path=str(CKPT_PATH),
)

model = DABiGRUCNNForecaster(model_cfg)
print(f"[model] n_params = {model.n_params():,}")

trainer = Trainer(model, train_cfg, KINK_AAVE, KINK_COMPOUND,
                  mlflow_experiment=MLFLOW_EXPERIMENT)

t0 = time.time()
try:
    fit_out = trainer.fit(train_loader, val_loader)
except RuntimeError as exc:
    if "CUDA out of memory" in str(exc):
        raise RuntimeError(
            "OOM - try Runtime -> Restart and reduce BATCH from 64 to 32 in cell 7, "
            "or pick a higher-VRAM GPU (A100/H100)."
        ) from exc
    raise
elapsed = time.time() - t0
print(f"\n[train] done in {elapsed/60:.1f} min   best_val_loss={fit_out['best_val_loss']:.4f}")
print(f"[train] checkpoint: {fit_out['ckpt']}")
READY_train = True

## 9. Validation metrics

Per PROJECT_2_PLAN.md S9.4: weighted Pearson per protocol on val, plus directional accuracy on the binary `r_aave > r_compound at t+12h` task (plan requires >= 55%).

In [ ]:
READY_metrics = False
assert READY_train, "Cell 8 (train) failed - fix that first."

import numpy as np
from forecaster.train import reconstruct_rate

best_model = trainer.model
best_model.train(False)  # inference mode; equivalent to .eval() but explicit

def _collect(loader):
    preds_, trues_ = [], []
    with torch.no_grad():
        for x_a, x_b, y in loader:
            x_a = x_a.to(device, non_blocking=True)
            x_b = x_b.to(device, non_blocking=True)
            out = best_model(x_a, x_b)
            r_hat = reconstruct_rate(out, KINK_AAVE, KINK_COMPOUND)
            preds_.append(r_hat.cpu().numpy())
            trues_.append(y.numpy())
    return np.concatenate(preds_), np.concatenate(trues_)

def weighted_pearson(y, yhat, w=None):
    y = np.asarray(y); yhat = np.asarray(yhat)
    w = np.ones_like(y) if w is None else np.asarray(w)
    my = np.average(y, weights=w); mp = np.average(yhat, weights=w)
    cov = np.average((y - my) * (yhat - mp), weights=w)
    vy  = np.average((y - my) ** 2, weights=w)
    vp  = np.average((yhat - mp) ** 2, weights=w)
    return float(cov / (np.sqrt(vy * vp) + 1e-12))

def r2_score(y, yhat):
    y = np.asarray(y); yhat = np.asarray(yhat)
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)

val_preds, val_trues = _collect(val_loader)
metrics = {
    "val/wpearson_aave":     weighted_pearson(val_trues[:, 0], val_preds[:, 0]),
    "val/wpearson_compound": weighted_pearson(val_trues[:, 1], val_preds[:, 1]),
    "val/r2_aave":           r2_score(val_trues[:, 0], val_preds[:, 0]),
    "val/r2_compound":       r2_score(val_trues[:, 1], val_preds[:, 1]),
    "val/dir_acc_aave_gt_compound": float(
        ((val_preds[:, 0] > val_preds[:, 1]) == (val_trues[:, 0] > val_trues[:, 1])).mean()
    ),
    "val/n_samples": int(val_preds.shape[0]),
    "train/best_val_loss": float(fit_out["best_val_loss"]),
    "train/elapsed_min": float(elapsed / 60),
}
if test_loader is not None:
    test_preds, test_trues = _collect(test_loader)
    metrics.update({
        "test/wpearson_aave":     weighted_pearson(test_trues[:, 0], test_preds[:, 0]),
        "test/wpearson_compound": weighted_pearson(test_trues[:, 1], test_preds[:, 1]),
        "test/r2_aave":           r2_score(test_trues[:, 0], test_preds[:, 0]),
        "test/r2_compound":       r2_score(test_trues[:, 1], test_preds[:, 1]),
        "test/dir_acc_aave_gt_compound": float(
            ((test_preds[:, 0] > test_preds[:, 1]) == (test_trues[:, 0] > test_trues[:, 1])).mean()
        ),
        "test/n_samples": int(test_preds.shape[0]),
    })

for k, v in metrics.items():
    print(f"  {k:38s} = {v}")

if metrics["val/dir_acc_aave_gt_compound"] < 0.55:
    print("\n[note] val dir-acc below the plan S5.1 acceptance threshold (0.55). "
          "Consider tuning hyperparameters (hidden_dim, dropout) and re-running.")
READY_metrics = True

## 10. Export to ONNX + verify torch <-> ONNX parity

`forecaster.export_onnx.export` does the conversion AND a numerical parity check at `atol=1e-4`. The output is what the backtest loads at runtime via onnxruntime (decouples backtest process from torch).

In [ ]:
READY_onnx = False
assert READY_metrics, "Cell 9 (metrics) failed - fix that first."

from forecaster.export_onnx import ExportConfig, export

ONNX_PATH = CKPT_DIR / "dual_branch_kink.onnx"
exp_cfg = ExportConfig(
    ckpt_path=CKPT_PATH,
    onnx_path=ONNX_PATH,
    opset=17,
    atol=1e-4,
    rtol=1e-4,
)
try:
    export(exp_cfg)
    print(f"[onnx] {ONNX_PATH}  ({ONNX_PATH.stat().st_size / 1024:.1f} KB)")
    metrics["onnx_path"] = str(ONNX_PATH)
    metrics["onnx_size_kb"] = ONNX_PATH.stat().st_size / 1024
    READY_onnx = True
except AssertionError as exc:
    print(f"[onnx] parity check FAILED: {exc}")
    print("[onnx] checkpoint is still saved; you can re-export locally and inspect.")
except Exception as exc:  # noqa: BLE001
    print(f"[onnx] export error: {exc}")

## 11. Persist outputs to Drive

Already-on-Drive: `CKPT_DIR` points at `MyDrive/.../trained_models/`. We additionally write a `metrics.json` next to the artifacts.

In [ ]:
import json, zipfile
from pathlib import Path

METRICS_PATH = CKPT_DIR / "metrics.json"
METRICS_PATH.write_text(json.dumps(metrics, indent=2, default=str))
print(f"[save] metrics -> {METRICS_PATH}")

print("\nArtifacts in", CKPT_DIR, ":")
for p in sorted(CKPT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.stat().st_size/1024:>10.1f} KB   {p.name}")

# When Drive is NOT mounted (VS Code -> Colab runtime) the artifacts live on
# the ephemeral VM. Bundle them into ONE zip on /content so they survive a
# single download and are trivial to pull back through VS Code.
if IN_COLAB and not DRIVE_OK:
    OUT_ZIP = Path("/content/predictive-mcdm-defi-trained.zip")
    with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(CKPT_DIR.iterdir()):
            if p.is_file():
                zf.write(p, arcname=p.name)
    print(f"\n[retrieve] Drive not mounted - bundled outputs into:")
    print(f"           {OUT_ZIP}  ({OUT_ZIP.stat().st_size/1024:.1f} KB)")
    print( "[retrieve] Pull it back via VS Code (any one of):")
    print( "   - VS Code Explorer on the remote: right-click the file -> Download, OR")
    print( "   - run in a notebook cell:  from google.colab import files; "
           "files.download('/content/predictive-mcdm-defi-trained.zip')   "
           "(works only from the Colab web tab, not VS Code), OR")
    print( "   - terminal/SFTP from the Colab side back to your machine.")
    print( "[retrieve] Then locally: unzip into forecaster/trained_models/ so")
    print( "           forecaster/trained_models/dual_branch_kink.onnx exists,")
    print( "           and run:  make finish")
else:
    print("\n[retrieve] Artifacts on Drive at "
          f"{CKPT_DIR} - copy dual_branch_kink.onnx + da_bigru_cnn.pt + "
          "metrics.json back to your local repo, then run `make finish`.")

## 12. Download instructions

After this notebook finishes, copy the artifacts from Drive back to your local repo:

```
MyDrive/predictive-mcdm-defi/trained_models/
    dual_branch_kink.onnx     ->   forecaster/trained_models/dual_branch_kink.onnx
    da_bigru_cnn.pt           ->   forecaster/trained_models/da_bigru_cnn.pt
    metrics.json              ->   results/colab_metrics.json  (or wherever you track results)
```

Then verify locally:

```powershell
.venv\Scripts\python -m forecaster.export_onnx --ckpt forecaster/trained_models/da_bigru_cnn.pt
```

(re-runs the parity check on your CPU, confirms portability)

Then the backtest will pick up the new ONNX automatically:

```powershell
.venv\Scripts\python -m backtest.run_main
```

This forecaster feeds into the H1 hypothesis test (PROJECT_2_PLAN.md S16): forecast-driven MCDM beats reactive EMA by >= 0.2 Sharpe over the 4-month test window.